In [7]:
import boto3
import pandas as pd
from botocore.config import Config

S3_ENDPOINT = 'https://seaweedfs:8333'
S3_REGION = 'us-east-1'
S3_CA_BUNDLE = '/etc/seaweedfs/tls/ca.crt'

s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id='admin',
    aws_secret_access_key='secret2',
    region_name=S3_REGION,
    verify=S3_CA_BUNDLE,
    config=Config(s3={'addressing_style': 'path'}),
)

In [8]:
bucket_response = s3.list_buckets()

buckets = pd.DataFrame(
    [
        {
            'Name': bucket['Name'],
            'CreationDate': bucket['CreationDate'],
        }
        for bucket in bucket_response.get('Buckets', [])
    ],
    columns=['Name', 'CreationDate'],
)

display(buckets)

,Name,CreationDate
0,my-bucket,2026-08-05 20:21:34+00:00
1,trino-lakehouse,2026-09-03 18:43:24+00:00


In [9]:
bucket_name = 'trino-lakehouse'
objects = []

for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket_name):
    objects.extend(
        {
            'Key': item['Key'],
            'Size': item['Size'],
            'LastModified': item['LastModified'],
        }
        for item in page.get('Contents', [])
    )

bucket_objects = pd.DataFrame(
    objects,
    columns=['Key', 'Size', 'LastModified'],
).sort_values('Key', ignore_index=True)

display(bucket_objects)

,Key,Size,LastModified
0,delta-codex-smoke/,0,2026-09-03 18:44:02+00:00
1,delta-codex-smoke/format_test-ef9867f15a4940a2...,345,2026-09-03 18:44:02+00:00
2,delta-codex-smoke/format_test-ef9867f15a4940a2...,1143,2026-09-03 18:44:03+00:00
3,delta-codex-smoke/format_test-ef9867f15a4940a2...,202,2026-09-03 18:44:03+00:00
4,test.db/,0,2026-09-04 14:33:27+00:00
5,test.db/my_table-83f43e1c9ed94e1996981c7a9501f...,698,2026-09-04 14:43:03+00:00
6,test.db/my_table-83f43e1c9ed94e1996981c7a9501f...,1775,2026-09-04 14:33:39+00:00
7,test.db/my_table-83f43e1c9ed94e1996981c7a9501f...,3708,2026-09-04 14:43:03+00:00
8,test.db/my_table-83f43e1c9ed94e1996981c7a9501f...,975,2026-09-04 14:43:03+00:00
9,test.db/my_table-83f43e1c9ed94e1996981c7a9501f...,7237,2026-09-04 14:43:03+00:00
